# Ajuste de hiperparámetros de los modelos finalistas

Finalistas definidos antes de este ajuste:

1. M3 con regresión logística.
2. M4 con regresión logística.
3. M3 con Histogram Gradient Boosting.

La búsqueda y selección se hace exclusivamente con validación cruzada agrupada por conglomerado en ENDES 2024. ENDES 2025 se utiliza después como evaluación temporal descriptiva. La métrica primaria es **PPV@20 %**.

También se reconstruyen M1 y el M2 original como referencias. En M2 se conservan edad, controles, atraso y presentaciones para poder compararlo con el trabajo anterior; esta redundancia se admite solo como línea base, no en los candidatos nuevos. La imputación del M2 se corrige para aprenderse únicamente en entrenamiento.

In [ ]:
from pathlib import Path
from time import perf_counter
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler

warnings.filterwarnings("ignore")
SEMILLA = 2026
N_BOOTSTRAP = 1_000

RAIZ = Path.cwd().resolve()
if RAIZ.name.lower() == "modelo":
    RAIZ = RAIZ.parent
if not (RAIZ / "datos_procesados").exists():
    raise FileNotFoundError("Ejecuta el notebook desde la raíz del proyecto o desde modelo/.")

RUTA_DATOS = RAIZ / "datos_procesados" / "tabla_modelo_2024_2025_limpia.csv"
RUTA_RESULTADOS = RAIZ / "modelo" / "resultados_ajuste_hiperparametros"
print(RUTA_DATOS)

## 1. Cohorte, target y variables

In [ ]:
datos = pd.read_csv(RUTA_DATOS, low_memory=False)
cohorte = datos.loc[
    datos["en_cohorte_modelo"].eq(True) & datos["tomo_7d"].isin([0, 1])
].copy()
cohorte["no_tomo_7d"] = (1 - cohorte["tomo_7d"]).astype("int8")
display(cohorte.groupby("anio")["no_tomo_7d"].agg(n="size", prevalencia="mean").style.format({"prevalencia": "{:.1%}"}))

PRESENTACIONES = ["recibio_jarabe", "recibio_gotas", "recibio_polvo"]
VARIABLES_M1 = ["edad_meses"]
VARIABLES_M2_REFERENCIA = ["edad_meses", "controles_cred_acum", "atraso_cred", *PRESENTACIONES]
VARIABLES_M3 = ["edad_meses", "atraso_cred", *PRESENTACIONES, "n_cred_con_fecha", "lugar_cred_codigo"]
VARIABLES_M4 = [
    *VARIABLES_M3, "anemia_oms", "edad_madre", "nivel_educativo_madre",
    "cuidador_habitual", "quintil_riqueza", "n_menores_5_hogar",
    "personas_por_dormitorio", "area",
]
BLOQUES = {"M1": VARIABLES_M1, "M2_referencia": VARIABLES_M2_REFERENCIA, "M3": VARIABLES_M3, "M4": VARIABLES_M4}
NUMERICAS = {
    "controles_cred_acum", "atraso_cred", "n_cred_con_fecha",
    "edad_madre", "n_menores_5_hogar", "personas_por_dormitorio"
}
BINARIAS = set(PRESENTACIONES)
CATEGORICAS = {
    "lugar_cred_codigo", "anemia_oms", "nivel_educativo_madre",
    "cuidador_habitual", "quintil_riqueza", "area"
}

assert len(VARIABLES_M4) == len(set(VARIABLES_M4))
assert "controles_cred_acum" not in VARIABLES_M3 + VARIABLES_M4
assert not any(v.startswith("tomo") or v.startswith("razon") for v in set(VARIABLES_M4))

## 2. Métricas y preparación sin fuga de información

In [ ]:
def agregar_atraso_cred(entrenamiento, evaluacion):
    entrenamiento = entrenamiento.copy()
    evaluacion = evaluacion.copy()
    mediana_edad = entrenamiento.groupby("edad_meses")["controles_cred_acum"].median()
    mediana_global = entrenamiento["controles_cred_acum"].median()
    for tabla in (entrenamiento, evaluacion):
        referencia = tabla["edad_meses"].map(mediana_edad).fillna(mediana_global)
        tabla["atraso_cred"] = tabla["controles_cred_acum"] - referencia
    return entrenamiento, evaluacion


def ppv_en_fraccion(y, puntaje, fraccion):
    y = np.asarray(y, dtype=float)
    puntaje = np.asarray(puntaje, dtype=float)
    n_elegir = max(1, int(np.ceil(fraccion * len(y))))
    umbral = np.partition(puntaje, len(puntaje) - n_elegir)[len(puntaje) - n_elegir]
    superiores = puntaje > umbral
    empatados = puntaje == umbral
    faltan = n_elegir - superiores.sum()
    positivos = y[superiores].sum()
    if faltan > 0:
        positivos += faltan * y[empatados].mean()
    return float(positivos / n_elegir)


def calibracion(y, puntaje):
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(puntaje, dtype=float), 1e-6, 1 - 1e-6)
    if np.std(p) < 1e-12 or len(np.unique(y)) < 2:
        return np.nan, np.nan
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    ajuste = LogisticRegression(C=1e6, max_iter=2_000).fit(logit, y)
    return float(ajuste.intercept_[0]), float(ajuste.coef_[0, 0])


def metricas(y, puntaje):
    y = np.asarray(y, dtype=int)
    puntaje = np.asarray(puntaje, dtype=float)
    intercepto, pendiente = calibracion(y, puntaje)
    return {
        "n": len(y), "prevalencia": y.mean(),
        "ppv10": ppv_en_fraccion(y, puntaje, 0.10),
        "ppv20": ppv_en_fraccion(y, puntaje, 0.20),
        "ppv30": ppv_en_fraccion(y, puntaje, 0.30),
        "auc": roc_auc_score(y, puntaje) if len(np.unique(y)) == 2 else np.nan,
        "brier": brier_score_loss(y, puntaje),
        "prediccion_media": puntaje.mean(),
        "calibracion_intercepto": intercepto,
        "calibracion_pendiente": pendiente,
    }

In [ ]:
def crear_preprocesador(variables, familia, n_knots=5):
    es_logistica = familia == "logistica"
    transformadores = []
    if "edad_meses" in variables:
        pasos_edad = [("imputar", SimpleImputer(strategy="median"))]
        if es_logistica:
            pasos_edad.append(("spline", SplineTransformer(
                n_knots=n_knots, degree=3, include_bias=False
            )))
        transformadores.append(("edad", Pipeline(pasos_edad), ["edad_meses"]))

    numericas = [v for v in variables if v in NUMERICAS]
    binarias = [v for v in variables if v in BINARIAS]
    categoricas = [v for v in variables if v in CATEGORICAS]
    if numericas:
        pasos_num = [("imputar", SimpleImputer(strategy="median", add_indicator=True))]
        if es_logistica:
            pasos_num.append(("escalar", StandardScaler()))
        transformadores.append(("num", Pipeline(pasos_num), numericas))
    if binarias:
        transformadores.append(("bin", SimpleImputer(strategy="most_frequent", add_indicator=True), binarias))
    if categoricas:
        transformadores.append((
            "cat", Pipeline([
                ("imputar", SimpleImputer(strategy="most_frequent", add_indicator=True)),
                ("onehot", OneHotEncoder(
                    handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False
                )),
            ]), categoricas
        ))
    return ColumnTransformer(transformadores, remainder="drop")


def crear_logistica(variables, parametros):
    penalizacion = parametros["penalty"]
    if penalizacion == "l2":
        estimador = LogisticRegression(
            C=parametros["C"], penalty="l2", solver="lbfgs", max_iter=4_000
        )
    elif penalizacion == "l1":
        estimador = LogisticRegression(
            C=parametros["C"], penalty="l1", solver="saga",
            max_iter=4_000, tol=1e-3, random_state=SEMILLA
        )
    else:
        estimador = LogisticRegression(
            C=parametros["C"], penalty="elasticnet", solver="saga",
            l1_ratio=parametros["l1_ratio"], max_iter=4_000, tol=1e-3,
            random_state=SEMILLA
        )
    return Pipeline([
        ("pre", crear_preprocesador(variables, "logistica", parametros["n_knots"])),
        ("modelo", estimador),
    ])


def crear_boosting(variables, parametros):
    estimador = HistGradientBoostingClassifier(
        learning_rate=parametros["learning_rate"], max_iter=200,
        max_leaf_nodes=parametros["max_leaf_nodes"],
        min_samples_leaf=parametros["min_samples_leaf"],
        l2_regularization=parametros["l2_regularization"],
        random_state=SEMILLA
    )
    return Pipeline([
        ("pre", crear_preprocesador(variables, "boosting")),
        ("modelo", estimador),
    ])


def crear_candidato(nombre, parametros):
    if nombre == "M3_logistica":
        return crear_logistica(VARIABLES_M3, parametros), VARIABLES_M3
    if nombre == "M4_logistica":
        return crear_logistica(VARIABLES_M4, parametros), VARIABLES_M4
    if nombre == "M3_gradient_boosting":
        return crear_boosting(VARIABLES_M3, parametros), VARIABLES_M3
    raise ValueError(nombre)

## 3. Espacio de búsqueda

La cuadrícula es deliberadamente pequeña y reproducible. No se ajustan umbrales porque PPV@20 siempre prioriza el 20 % superior.

In [ ]:
CONFIGS_LOGISTICA = []
for n_knots in [3, 5, 7]:
    for C in [0.03, 0.1, 0.3, 1.0, 3.0]:
        CONFIGS_LOGISTICA.append({"n_knots": n_knots, "penalty": "l2", "C": C})
    for C in [0.1, 0.3, 1.0]:
        CONFIGS_LOGISTICA.append({"n_knots": n_knots, "penalty": "l1", "C": C})
    for C in [0.1, 0.3]:
        for l1_ratio in [0.25, 0.5, 0.75]:
            CONFIGS_LOGISTICA.append({
                "n_knots": n_knots, "penalty": "elasticnet",
                "C": C, "l1_ratio": l1_ratio,
            })

CONFIGS_BOOSTING = [
    {
        "learning_rate": learning_rate, "max_leaf_nodes": max_leaf_nodes,
        "min_samples_leaf": min_samples_leaf, "l2_regularization": l2_regularization,
    }
    for learning_rate in [0.03, 0.07]
    for max_leaf_nodes in [7, 15]
    for min_samples_leaf in [20, 40]
    for l2_regularization in [0.0, 1.0]
]
CONFIGURACIONES = {
    "M3_logistica": CONFIGS_LOGISTICA,
    "M4_logistica": CONFIGS_LOGISTICA,
    "M3_gradient_boosting": CONFIGS_BOOSTING,
}
print(f"Configuraciones logísticas por modelo: {len(CONFIGS_LOGISTICA)}")
print(f"Configuraciones de boosting: {len(CONFIGS_BOOSTING)}")
print(f"Ajustes totales contando 5 folds: {sum(len(v) for v in CONFIGURACIONES.values()) * 5:,}")

## 4. Preparar cinco folds idénticos para todos los candidatos

In [ ]:
desarrollo = cohorte.loc[cohorte["anio"] == 2024].reset_index(drop=True)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
folds_preparados = []

for fold, (i_entrena, i_valida) in enumerate(
    cv.split(desarrollo, desarrollo["no_tomo_7d"], groups=desarrollo["conglomerado"]),
    start=1,
):
    entrena = desarrollo.iloc[i_entrena].copy()
    valida = desarrollo.iloc[i_valida].copy()
    entrena, valida = agregar_atraso_cred(entrena, valida)
    folds_preparados.append((fold, entrena, valida))

print(f"Folds preparados: {len(folds_preparados)}")

## 5. Búsqueda de hiperparámetros en 2024

In [ ]:
filas_busqueda = []
inicio_total = perf_counter()

for candidato, configuraciones in CONFIGURACIONES.items():
    print(f"Iniciando {candidato}: {len(configuraciones)} configuraciones")
    for numero, parametros in enumerate(configuraciones, start=1):
        config_id = f"{candidato}__{numero:03d}"
        parametros_json = json.dumps(parametros, sort_keys=True)
        for fold, entrena, valida in folds_preparados:
            modelo, variables = crear_candidato(candidato, parametros)
            y_entrena = entrena["no_tomo_7d"].to_numpy()
            y_valida = valida["no_tomo_7d"].to_numpy()
            inicio = perf_counter()
            modelo.fit(entrena[variables], y_entrena)
            p = modelo.predict_proba(valida[variables])[:, 1]
            segundos = perf_counter() - inicio
            filas_busqueda.append({
                "candidato": candidato, "config_id": config_id,
                "parametros": parametros_json, "fold": fold,
                "segundos": segundos, **metricas(y_valida, p),
            })
        if numero % 10 == 0 or numero == len(configuraciones):
            print(f"  {numero}/{len(configuraciones)} completadas")

print(f"Búsqueda terminada en {(perf_counter() - inicio_total) / 60:.1f} minutos")
resultados_busqueda_folds = pd.DataFrame(filas_busqueda)

In [ ]:
columnas_resumen = ["ppv10", "ppv20", "ppv30", "auc", "brier", "segundos"]
resultados_busqueda = resultados_busqueda_folds.groupby(
    ["candidato", "config_id", "parametros"]
)[columnas_resumen].agg(["mean", "std"])
resultados_busqueda.columns = [f"{m}_{e}" for m, e in resultados_busqueda.columns]
resultados_busqueda = resultados_busqueda.reset_index()

mejores_filas = []
for candidato, sub in resultados_busqueda.groupby("candidato"):
    mejor = sub.sort_values(
        ["ppv20_mean", "ppv20_std", "auc_mean", "brier_mean"],
        ascending=[False, True, False, True],
    ).iloc[0]
    mejores_filas.append(mejor)

mejores_configuraciones = pd.DataFrame(mejores_filas).sort_values(
    ["ppv20_mean", "auc_mean"], ascending=False
)
display(mejores_configuraciones[[
    "candidato", "parametros", "ppv10_mean", "ppv20_mean",
    "ppv20_std", "ppv30_mean", "auc_mean", "brier_mean"
]].style.format({c: "{:.3f}" for c in mejores_configuraciones.columns if c.endswith(("_mean", "_std"))}))

PARAMETROS_GANADORES = {
    fila["candidato"]: json.loads(fila["parametros"])
    for _, fila in mejores_configuraciones.iterrows()
}
ganador_cv = mejores_configuraciones.iloc[0]["candidato"]
print(f"Ganador según CV 2024: {ganador_cv}")

## 6. Evaluación temporal de las mejores configuraciones

M1 y M2 se incluyen como referencias. M2 conserva la estructura del modelo del compañero, pero sin utilizar medianas del conjunto de prueba.

In [ ]:
entrena_2024 = cohorte.loc[cohorte["anio"] == 2024].reset_index(drop=True)
prueba_2025 = cohorte.loc[cohorte["anio"] == 2025].reset_index(drop=True)
entrena_2024, prueba_2025 = agregar_atraso_cred(entrena_2024, prueba_2025)
y_entrena = entrena_2024["no_tomo_7d"].to_numpy()
y_prueba = prueba_2025["no_tomo_7d"].to_numpy()

parametros_base = {"n_knots": 5, "penalty": "l2", "C": 1.0}
modelos_finales = {
    "M1_edad": (crear_logistica(VARIABLES_M1, parametros_base), VARIABLES_M1),
    "M2_companero_corregido": (
        crear_logistica(VARIABLES_M2_REFERENCIA, parametros_base), VARIABLES_M2_REFERENCIA
    ),
}
for candidato, parametros in PARAMETROS_GANADORES.items():
    modelos_finales[candidato] = crear_candidato(candidato, parametros)

predicciones_2025 = {}
modelos_ajustados = {}
filas_temporales = []
for nombre, (modelo, variables) in modelos_finales.items():
    inicio = perf_counter()
    modelo.fit(entrena_2024[variables], y_entrena)
    p = modelo.predict_proba(prueba_2025[variables])[:, 1]
    segundos = perf_counter() - inicio
    modelos_ajustados[nombre] = modelo
    predicciones_2025[nombre] = p
    filas_temporales.append({"modelo": nombre, "segundos": segundos, **metricas(y_prueba, p)})

resultados_temporales = pd.DataFrame(filas_temporales).sort_values(
    ["ppv20", "auc", "brier"], ascending=[False, False, True]
)
display(resultados_temporales.style.format({
    "prevalencia": "{:.1%}", "ppv10": "{:.1%}",
    "ppv20": "{:.1%}", "ppv30": "{:.1%}",
    "auc": "{:.3f}", "brier": "{:.3f}",
    "prediccion_media": "{:.1%}", "calibracion_intercepto": "{:.3f}",
    "calibracion_pendiente": "{:.3f}", "segundos": "{:.2f}",
}))

## 7. Conglomerados de 2025 no vistos en 2024

In [ ]:
conglomerados_2024 = set(entrena_2024["conglomerado"].unique())
mascara_no_vistos = ~prueba_2025["conglomerado"].isin(conglomerados_2024)
y_no_vistos = y_prueba[mascara_no_vistos.to_numpy()]
resultados_no_vistos = pd.DataFrame([
    {"modelo": nombre, **metricas(y_no_vistos, p[mascara_no_vistos.to_numpy()])}
    for nombre, p in predicciones_2025.items()
]).sort_values(["ppv20", "auc"], ascending=False)
print(f"Niños en conglomerados no vistos: {mascara_no_vistos.sum():,} ({mascara_no_vistos.mean():.1%})")
display(resultados_no_vistos[["modelo", "n", "ppv20", "auc", "brier", "prediccion_media"]].style.format({
    "ppv20": "{:.1%}", "auc": "{:.3f}", "brier": "{:.3f}",
    "prediccion_media": "{:.1%}",
}))

## 8. Comparaciones directas con bootstrap por conglomerados

In [ ]:
comparaciones = [
    ("M3_logistica", "M2_companero_corregido"),
    ("M4_logistica", "M2_companero_corregido"),
    ("M3_gradient_boosting", "M2_companero_corregido"),
    ("M4_logistica", "M3_logistica"),
    ("M3_gradient_boosting", "M3_logistica"),
]
rng = np.random.default_rng(SEMILLA)
grupos = prueba_2025["conglomerado"].to_numpy()
grupos_unicos = np.unique(grupos)
indices_grupo = {g: np.flatnonzero(grupos == g) for g in grupos_unicos}
filas_bootstrap = []

for modelo_a, modelo_b in comparaciones:
    dif_ppv, dif_auc = [], []
    for _ in range(N_BOOTSTRAP):
        muestra = rng.choice(grupos_unicos, size=len(grupos_unicos), replace=True)
        indices = np.concatenate([indices_grupo[g] for g in muestra])
        y_b = y_prueba[indices]
        p_a = predicciones_2025[modelo_a][indices]
        p_b = predicciones_2025[modelo_b][indices]
        dif_ppv.append(ppv_en_fraccion(y_b, p_a, 0.20) - ppv_en_fraccion(y_b, p_b, 0.20))
        dif_auc.append(roc_auc_score(y_b, p_a) - roc_auc_score(y_b, p_b))

    for nombre_metrica, valores in [("ppv20", dif_ppv), ("auc", dif_auc)]:
        estimacion = (
            ppv_en_fraccion(y_prueba, predicciones_2025[modelo_a], 0.20) -
            ppv_en_fraccion(y_prueba, predicciones_2025[modelo_b], 0.20)
            if nombre_metrica == "ppv20" else
            roc_auc_score(y_prueba, predicciones_2025[modelo_a]) -
            roc_auc_score(y_prueba, predicciones_2025[modelo_b])
        )
        filas_bootstrap.append({
            "modelo_a": modelo_a, "modelo_b": modelo_b, "metrica": nombre_metrica,
            "diferencia": estimacion, "ic95_inferior": np.percentile(valores, 2.5),
            "ic95_superior": np.percentile(valores, 97.5),
        })

resultados_bootstrap = pd.DataFrame(filas_bootstrap)
display(resultados_bootstrap.style.format({
    "diferencia": "{:+.3f}", "ic95_inferior": "{:+.3f}",
    "ic95_superior": "{:+.3f}",
}))

## 9. Equidad del ganador de CV 2024

Esta auditoría no cambia la selección de hiperparámetros; sirve para detectar diferencias importantes entre grupos.

In [ ]:
auditoria = prueba_2025.copy()
auditoria["puntaje"] = predicciones_2025[ganador_cv]
filas_equidad = []
for variable in ["area", "nivel_educativo_madre", "quintil_riqueza", "lengua_materna"]:
    for grupo, sub in auditoria.groupby(variable, dropna=False):
        if len(sub) < 100:
            continue
        y_g = sub["no_tomo_7d"].to_numpy()
        p_g = sub["puntaje"].to_numpy()
        filas_equidad.append({
            "variable": variable, "grupo": str(grupo), "n": len(sub),
            "observado": y_g.mean(), "predicho": p_g.mean(),
            "ppv20": ppv_en_fraccion(y_g, p_g, 0.20),
            "auc": roc_auc_score(y_g, p_g) if len(np.unique(y_g)) == 2 else np.nan,
        })
resultados_equidad = pd.DataFrame(filas_equidad)
print(f"Modelo auditado: {ganador_cv}")
display(resultados_equidad.style.format({
    "observado": "{:.1%}", "predicho": "{:.1%}",
    "ppv20": "{:.1%}", "auc": "{:.3f}",
}))

## 10. Guardar resultados

Este notebook no reajusta todavía con 2024 + 2025 ni exporta un modelo de producción.

In [ ]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
resultados_busqueda_folds.to_csv(RUTA_RESULTADOS / "busqueda_folds_2024.csv", index=False)
resultados_busqueda.to_csv(RUTA_RESULTADOS / "busqueda_resumen_2024.csv", index=False)
mejores_configuraciones.to_csv(RUTA_RESULTADOS / "mejores_configuraciones_2024.csv", index=False)
resultados_temporales.to_csv(RUTA_RESULTADOS / "evaluacion_temporal_2025.csv", index=False)
resultados_no_vistos.to_csv(RUTA_RESULTADOS / "conglomerados_no_vistos_2025.csv", index=False)
resultados_bootstrap.to_csv(RUTA_RESULTADOS / "comparaciones_bootstrap_2025.csv", index=False)
resultados_equidad.to_csv(RUTA_RESULTADOS / "equidad_ganador_cv_2025.csv", index=False)
(RUTA_RESULTADOS / "mejores_parametros.json").write_text(
    json.dumps(PARAMETROS_GANADORES, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Ganador de CV 2024: {ganador_cv}")
print(f"Resultados: {RUTA_RESULTADOS}")

## Regla de decisión posterior

La configuración ganadora se define solo con CV 2024. Para recomendar un modelo operativo también se revisarán estabilidad temporal, conglomerados no vistos, calibración, equidad y la comparación directa contra M2. Como 2025 ya se ha consultado en varias etapas, la confirmación final deberá realizarse con datos nuevos del piloto.